# HybridDocling OCR Benchmark

**Benchmark: Confidence-Gated Hybrid OCR with Full Docling Pipeline**

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/buinguyenkhai/stock-report-agent-20251.git
%cd stock-report-agent-20251

In [ ]:
# Install system dependencies (Tesseract + Vietnamese language pack)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-vie libtesseract-dev libleptonica-dev

# Install Python dependencies
!pip install -q -r "requirements.txt"
!pip install -q pymupdf
!pip install -q "docling[tesserocr]"
!pip install -q marker-pdf surya-ocr

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set environment variables
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# Test imports work
import sys
sys.path.insert(0, '.')

# Direct import to avoid dependency issues
from evaluation.ocr_benchmark import PageLevelBenchmark
from services.ocr.hybrid_pdf_pipeline import HybridPdfPipeline
print("Imports OK")
print(f"HybridPdfPipeline: {HybridPdfPipeline}")

## 2. Full HybridDocling Benchmark (~10 hours)

This benchmark uses:
- **HybridOcrModel**: Confidence-gated routing between Tesseract (fast) and Surya (accurate)
- **Full Docling Pipeline**: Layout detection + Table structure recognition + Markdown export
- **HybridPdfPipeline**: Custom pipeline that injects HybridOcrModel into Docling

In [ ]:
# Run full benchmark with HybridDocling
!python -m evaluation.ocr_benchmark.page_level_benchmark \
    --engine hybrid_docling \
    --save-outputs \
    --output results/hybrid_docling_full.json

## 3. Results Analysis

In [ ]:
import json

# Load results
with open('results/hybrid_docling_full.json', 'r') as f:
    results = json.load(f)

print("="*60)
print("HybridDocling Full Benchmark Results")
print("="*60)
print(f"\nTotal companies: {results['total_companies']}")
print(f"Total pages: {results['total_pages']}")
print(f"Successful pages: {results['successful_pages']}")
print(f"\n--- Overall Metrics ---")
print(f"NumF1: {results.get('overall_aggregated_number_f1', 'N/A')*100:.2f}%")
print(f"Word Recall: {results.get('overall_aggregated_word_recall', 'N/A')*100:.2f}%")
print(f"\n--- Per-Page Averages (mean \u00b1 std) ---")
print(f"NumF1: {results['overall_avg_number_f1']*100:.2f}% \u00b1 {results['overall_std_number_f1']*100:.2f}%")
print(f"Word Recall: {results['overall_avg_content_word_recall']*100:.2f}% \u00b1 {results['overall_std_content_word_recall']*100:.2f}%")
print(f"FA-CER: {results['overall_avg_format_agnostic_cer']*100:.2f}% \u00b1 {results['overall_std_format_agnostic_cer']*100:.2f}%")

In [ ]:
# Check peak VRAM usage per page
vram_data = []
for company in results.get('company_results', []):
    for page in company.get('page_results', []):
        if page.get('peak_vram_mb'):
            vram_data.append(page['peak_vram_mb'])

if vram_data:
    import numpy as np
    print(f"\n--- VRAM Usage ---")
    print(f"Average peak VRAM: {np.mean(vram_data):.2f} MB")
    print(f"Max peak VRAM: {np.max(vram_data):.2f} MB")
else:
    print("No VRAM data recorded")

In [ ]:
!zip -r hybrid_docling.zip results